In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# 1. تجهيز بيانات سورة الفاتحة (نصوص كاملة لأن BERT يفهم السياق)
sentences_text = [
    "بسم الله الرحمن الرحيم",
    "الحمد لله رب العالمين",
    "الرحمن الرحيم",
    "مالك يوم الدين",
    "إياك نعبد وإياك نستعين",
    "اهدنا الصراط المستقيم",
    "صراط الذين أنعمت عليهم غير المغضوب عليهم ولا الضالين"
]

# 2. تحميل نموذج AraBERT (أسرع وأدق للغة العربية)
model_name = "aubmindlab/bert-base-arabertv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 3. دالة لتحويل النص إلى متجه (Embedding) باستخدام BERT
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # نأخذ متوسط الطبقة الأخيرة لتمثيل الجملة بمتجه واحد
    return outputs.last_hidden_state.mean(dim=1).numpy()

# 4. استخراج المتجهات لجميع الآيات
embeddings = [get_bert_embedding(s) for s in sentences_text]

# 5. الكلمة أو الآية التي نريد البحث عن تشابه معها
query = "الرحمن"
query_embedding = get_bert_embedding(query)

# 6. حساب التشابه بين "الرحمن" وكل آية في السورة
similarities = []
for i, emb in enumerate(embeddings):
    score = cosine_similarity(query_embedding, emb)[0][0]
    similarities.append((sentences_text[i], score))

# 7. تحويل النتيجة إلى جدول مرتب
df = pd.DataFrame(similarities, columns=['الآية/النص', 'درجة التشابه السياقي'])
df = df.sort_values(by='درجة التشابه السياقي', ascending=False)

print(f"\nدرجة تشابه الآيات مع كلمة: {query} (بناءً على سياق BERT)\n")
print(df.to_string(index=False))